In [ ]:
import pandas as pd
import os, ssl

In [ ]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

hallmark_url = "https://github.com/osmanbeyoglulab/STAN/raw/main/priors/hallmark.csv"
hallmark = pd.read_csv(hallmark_url, index_col=0)

In [ ]:
def make_dir(dir):
    if not os.path.exists(dir):
       # Create a new directory because it does not exist
       os.makedirs(dir)
        
def load_hallmark(source_dir="data/gene_pw/"):
    fname = source_dir+"hallmark.csv"
    if os.path.isfile(fname):
        hallmark = pd.read_csv(fname, index_col=0)
    else:
        print('Downloading MSigDB database...')
        hallmark_url = "https://github.com/osmanbeyoglulab/STAN/raw/main/priors/hallmark.csv"
        hallmark = pd.read_csv(hallmark_url, index_col=0)
        make_dir(source_dir)
        hallmark.to_csv(fname)

    hallmark = hallmark.loc[:, ['gs_name', 'gene_symbol']]
    hallmark.columns = ["pathway", "gene"]
    return hallmark

In [ ]:
def make_gene_pw_matrix(source="hallmark",
                        gene_universe=None,
#                         min_pws_per_gene=10,
#                         min_genes_per_pw=10,
                        source_dir: str = "data/gene_pw/"):

    if type(source) == pd.DataFrame:
        gene_pw = source
    elif source == "hallmark":
        gene_pw = load_hallmark(source_dir=source_dir)

    if gene_universe is not None:
        gene_pw = gene_pw.query("gene in @gene_universe")

    gene_pw['values'] = 1
    gene_pw = gene_pw.pivot_table(index='gene', columns='pathway', aggfunc='mean', values='values', fill_value=0)

#     # linan's version
#     gene_sum = gene_pw.T.sum()
#     gene_pw.drop(gene_pw.index[gene_sum<min_pws_per_gene], axis=0, inplace=True)
#     pw_sum = gene_pw.sum()
#     gene_pw.drop(gene_pw.columns[pw_sum<min_genes_per_pw], axis=1, inplace=True)
    return gene_pw

In [ ]:
def add_gene_pw_matrix(
        data: AnnData,
        min_cells_proportion: float = 0.2,
#         min_pws_per_gene: int = 5,
#         min_genes_per_pw: int = 10,
        gene_pw_source: Union[Literal["progeny", "hallmark"], DataFrame] = "hallmark",
        source_dir: str = "data/gene_pw/"
        ) -> Union[AnnData, None]:

    adata = data.copy()
    pw_universe = adata.var_names.copy()
    sc.pp.filter_genes(adata, min_cells=min_cells_proportion*adata.n_obs)

    gene_pw = make_gene_pw_matrix(source = gene_pw_source,
                                gene_universe = adata.var_names,
#                                 min_pws_per_gene = min_pws_per_gene,
#                                 min_genes_per_pw = min_genes_per_pw,
                                source_dir = source_dir)

    adata = adata[:, gene_pw.index]
    adata.varm['gene_pw'] = gene_pw
    adata.uns['pw_names'] = gene_pw.columns.to_list()
    return adata

In [ ]:
import os
import pandas as pd
import scanpy as sc
from anndata import AnnData
from pandas import DataFrame
from typing import Optional, Union, Literal, List

In [ ]:
adata = sc.read_h5ad("data/gbm_qc/UKF243.h5ad")

In [ ]:
adata1 = add_gene_pw_matrix(
        adata,
        min_cells_proportion = 0.2,
#         min_pws_per_gene = 2,
#         min_genes_per_pw = 5,
        source_dir="data/gene_pw/")
adata1

In [ ]:
adata

In [ ]:
import gseapy as gp

msig = gp.Msigdb()
gmt = msig.get_gmt(category='h.all')

In [1]:
import gseapy as gp
gp.__version__


'1.1.6'

In [2]:
pip install --upgrade gseapy

DEPRECATION: Loading egg at /Users/zhanglinan/anaconda3/envs/bioinfo/lib/python3.11/site-packages/SEDR-1.0.0-py3.11.egg is deprecated. pip 23.3 will enforce this behaviour change. A possible replacement is to use pip for package installation..
  Obtaining dependency information for gseapy from https://files.pythonhosted.org/packages/9e/b2/21f09caa13605f3736994e3e2cc2bb8b5d66c39efbd46f508d9bc70feea8/gseapy-1.1.9-cp311-cp311-macosx_11_0_arm64.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 535.8/535.8 kB 39.3 kB/s eta 0:00:00a 0:00:01
  Attempting uninstall: gseapy
    Found existing installation: gseapy 1.1.6
    Uninstalling gseapy-1.1.6:
      Successfully uninstalled gseapy-1.1.6
Note: you may need to restart the kernel to use updated packages.
